# AIML339 — Train/Test Split

**Purpose:** Split the raw `Img` folder into `train/` and `test/`

I encode class folder names as:
- Digits: `0..9`
- Uppercase: `U_A` .. `U_Z`
- Lowercase: `L_a` .. `L_z`

In [1]:
# Config
SRC_DIR = r"C:\Users\1\Downloads\archive\Img"  # Source directory containing raw images
DST_DIR = r"C:\Users\1\Downloads\english_2split_case"  # Destination directory for train/test split

MAP_CHARS   = True  # Whether to map class names to case-safe labels
TRAIN_RATIO = 0.80  # Ratio of images to use for training
SEED        = 42  # Random seed for reproducibility

from pathlib import Path  # Import Path for handling file paths
SRC_DIR, DST_DIR  # Display source and destination directories

('C:\\Users\\1\\Downloads\\archive\\Img',
 'C:\\Users\\1\\Downloads\\english_2split_case')

In [2]:
# Imports & helpers
import os, re, random, shutil  # Import required libraries
from pathlib import Path  # Import Path again for file path handling

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.gif'}  # Supported image file extensions
DIGITS   = [*list('0123456789')]  # List of digit characters
UPPER    = [chr(c) for c in range(ord('A'), ord('Z')+1)]  # List of uppercase letters
LOWER    = [chr(c) for c in range(ord('a'), ord('z')+1)]  # List of lowercase letters
SEQ_62   = DIGITS + UPPER + LOWER  # Combined list of digits, uppercase, and lowercase letters

# Function to generate a natural sorting key for case-sensitive strings
def natural_key_case_sensitive(s):
    s = str(s)  # Convert input to string
    parts = re.split(r'(\d+)', s)  # Split string into parts (numbers and non-numbers)
    key = []  # Initialize sorting key
    for p in parts:  # Iterate over parts
        if p.isdigit(): key.append(int(p))  # Convert numeric parts to integers
        else: key.append(p)  # Keep non-numeric parts as strings
    return key  # Return the sorting key

# Function to find the root directory containing images
def find_img_root(src: Path) -> Path:
    cur = src  # Start with the source directory
    for _ in range(3):  # Check up to 3 levels deep
        if (cur / 'Img').is_dir() and any((cur / 'Img').iterdir()):  # Check if 'Img' folder exists and is not empty
            cur = cur / 'Img'  # Update current directory to 'Img'
        else:
            break  # Stop if no 'Img' folder is found
    return cur  # Return the detected image root directory

# Function to detect class directories containing images
def detect_class_dirs(img_root: Path):
    cand = [p for p in img_root.iterdir() if p.is_dir()]  # List all subdirectories
    class_dirs = []  # Initialize list of class directories
    for d in cand:  # Iterate over candidate directories
        has_img = any(f.is_file() and f.suffix.lower() in IMG_EXTS for f in d.iterdir())  # Check if directory contains images
        if has_img:
            class_dirs.append(d)  # Add directory to class_dirs if it contains images
    return sorted(class_dirs, key=lambda p: natural_key_case_sensitive(p.name))  # Return sorted class directories

# Function to group loose images into sample folders
def group_flat_images_into_samples(img_root: Path):
    files = list(img_root.glob('*'))  # List all files in the root directory
    imgs  = [f for f in files if f.is_file() and f.suffix.lower() in IMG_EXTS]  # Filter image files
    if not imgs:
        return 0  # Return 0 if no images are found
    pat = re.compile(r"img(\d{3})-\d+\.[a-zA-Z0-9]+$")  # Regex pattern to match image filenames
    moved = 0  # Counter for moved images
    for f in imgs:  # Iterate over image files
        m = pat.match(f.name)  # Match filename against the pattern
        if not m:
            continue  # Skip if no match
        sample = f"Sample{m.group(1)}"  # Extract sample number from filename
        dst_dir = img_root / sample  # Create destination directory path
        dst_dir.mkdir(exist_ok=True)  # Create destination directory if it doesn't exist
        shutil.move(str(f), str(dst_dir / f.name))  # Move image file to the destination directory
        moved += 1  # Increment moved counter
    return moved  # Return the number of moved images

# Function to generate case-safe labels for class names
def case_safe_label(lbl: str) -> str:
    if lbl in DIGITS: return lbl  # Return digits as-is
    if lbl in UPPER:  return f"U_{lbl}"  # Prefix uppercase letters with 'U_'
    if lbl in LOWER:  return f"L_{lbl}"  # Prefix lowercase letters with 'L_'
    return lbl  # Return other labels as-is

# Function to build a mapping of class directories to case-safe labels
def build_mapping(class_dirs, map_chars: bool):
    names = [d.name for d in class_dirs]  # Get names of class directories
    if map_chars and all(n.lower().startswith('sample') for n in names) and len(names) == 62:  # Check if mapping is needed
        def sample_num(n):
            m = re.search(r'(\d+)$', n)  # Extract sample number from name
            return int(m.group(1)) if m else 9999  # Return sample number or a large value if not found
        ordered = sorted(names, key=sample_num)  # Sort names by sample number
        raw_labels = SEQ_62  # Use predefined sequence of labels
        mapped = [case_safe_label(x) for x in raw_labels]  # Generate case-safe labels
        return {src: dst for src, dst in zip(ordered, mapped)}  # Return mapping of original to case-safe labels
    lower_map = {}  # Initialize mapping for other cases
    for n in names:  # Iterate over names
        if len(n) == 1 and n in UPPER + LOWER:  # Check if name is a single character
            lower_map[n] = case_safe_label(n)  # Map to case-safe label
        else:
            lower_map[n] = n  # Keep name as-is
    return lower_map  # Return the mapping

# Function to split files into train and test sets
def two_way_split(files, train_ratio=0.8, seed=42):
    rnd = random.Random(seed)  # Initialize random generator with seed
    files = list(files)  # Convert files to a list
    rnd.shuffle(files)  # Shuffle the files
    n = len(files)  # Get total number of files
    n_tr = int(round(n * train_ratio))  # Calculate number of training files
    return files[:n_tr], files[n_tr:]  # Return train and test splits

# Function to safely copy files to a destination directory
def safe_copy(files, dst_dir: Path):
    dst_dir.mkdir(parents=True, exist_ok=True)  # Create destination directory if it doesn't exist
    for f in files:  # Iterate over files
        shutil.copy2(f, dst_dir / f.name)  # Copy file to destination directory


In [3]:
# Split-only pipeline (case-safe)
src = Path(SRC_DIR)  # Convert source directory to Path object
dst = Path(DST_DIR)  # Convert destination directory to Path object
dst.mkdir(parents=True, exist_ok=True)  # Create destination directory if it doesn't exist

img_root = find_img_root(src)  # Find the root directory containing images
print('Image root:', img_root)  # Print the detected image root

loose_imgs = [p for p in img_root.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]  # Detect loose images
if loose_imgs:
    print(f"Detected {len(loose_imgs)} loose images; grouping into SampleXXX folders...")  # Print number of loose images
    moved = group_flat_images_into_samples(img_root)  # Group loose images into sample folders
    print(f"Moved {moved} images into SampleXXX folders.")  # Print number of moved images
else:
    print("No loose images detected; proceeding.")  # Print message if no loose images are found

class_dirs = detect_class_dirs(img_root)  # Detect class directories
if not class_dirs:
    raise RuntimeError(f"No class directories with images found under: {img_root}")  # Raise error if no class directories are found

mapping = build_mapping(class_dirs, MAP_CHARS)  # Build mapping of class directories to case-safe labels
print('Example mapping (first 12):')  # Print example mapping
for i,(k) in enumerate(mapping.keys()):  # Iterate over mapping keys
    if i>=12: break  # Stop after 12 examples
    print(' ', k, '->', mapping[k])  # Print mapping

(dst / 'train').mkdir(parents=True, exist_ok=True)  # Create train directory
(dst / 'test').mkdir(parents=True, exist_ok=True)  # Create test directory

for d in class_dirs:  # Iterate over class directories
    files = [p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXTS]  # Get image files in directory
    files.sort(key=lambda p: natural_key_case_sensitive(p.name))  # Sort files naturally
    tr, te = two_way_split(files, train_ratio=TRAIN_RATIO, seed=SEED)  # Split files into train and test sets
    dst_name = mapping[d.name]  # Get case-safe label for directory
    safe_copy(tr, dst / 'train' / dst_name)  # Copy training files to train directory
    safe_copy(te, dst / 'test'  / dst_name)  # Copy test files to test directory
    print(f"{d.name} → {dst_name}: {len(tr)} train, {len(te)} test")  # Print split details

print(f"\nDone. Two-way split created at: {dst}")  # Print completion message

Image root: C:\Users\1\Downloads\archive\Img
No loose images detected; proceeding.
Example mapping (first 12):
  Sample001 -> 0
  Sample002 -> 1
  Sample003 -> 2
  Sample004 -> 3
  Sample005 -> 4
  Sample006 -> 5
  Sample007 -> 6
  Sample008 -> 7
  Sample009 -> 8
  Sample010 -> 9
  Sample011 -> U_A
  Sample012 -> U_B
Sample001 → 0: 44 train, 11 test
Sample002 → 1: 44 train, 11 test
Sample003 → 2: 44 train, 11 test
Sample004 → 3: 44 train, 11 test
Sample005 → 4: 44 train, 11 test
Sample006 → 5: 44 train, 11 test
Sample007 → 6: 44 train, 11 test
Sample008 → 7: 44 train, 11 test
Sample009 → 8: 44 train, 11 test
Sample010 → 9: 44 train, 11 test
Sample011 → U_A: 44 train, 11 test
Sample012 → U_B: 44 train, 11 test
Sample013 → U_C: 44 train, 11 test
Sample014 → U_D: 44 train, 11 test
Sample015 → U_E: 44 train, 11 test
Sample016 → U_F: 44 train, 11 test
Sample017 → U_G: 44 train, 11 test
Sample018 → U_H: 44 train, 11 test
Sample019 → U_I: 44 train, 11 test
Sample020 → U_J: 44 train, 11 test
S